# MOFA-FLEX BASE - Timing (3 runs)

💡 **Environment:** `clamp-analyses`  

In [ ]:
library(here)
library(reticulate)
set.seed(1)

env_name <- "clamp-analyses"
if (!nzchar(Sys.getenv("RETICULATE_PYTHON"))) {
  conda <- Sys.which("conda")
  if (nzchar(conda)) {
    cmd <- sprintf('%s run -n %s python -c "import sys; print(sys.executable)"', shQuote(conda), shQuote(env_name))
    py <- tryCatch(system(cmd, intern = TRUE), error = function(e) character(0))
    if (length(py) == 1 && nzchar(py) && file.exists(py)) {
      Sys.setenv(RETICULATE_PYTHON = py)
    } else {
      Sys.setenv(RETICULATE_PYTHON = Sys.which("python"))
    }
  } else {
    Sys.setenv(RETICULATE_PYTHON = Sys.which("python"))
  }
}

In [ ]:
mfl <- import("mofaflex", delay_load = FALSE)
ad  <- import("anndata",  delay_load = FALSE)
np  <- import("numpy",    delay_load = FALSE)
pd  <- import("pandas",   delay_load = FALSE)

In [ ]:
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3

In [ ]:
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))
K <- readRDS(here("output/gtex/CLAMP_K_gtex.rds"))

X <- t(as.matrix(gtex_data))
X_np <- np$array(X, dtype = "float32")
adata <- ad$AnnData(X_np)
adata$obs_names <- pd$Index(colnames(gtex_data))
adata$var_names <- pd$Index(rownames(gtex_data))
data_list <- dict(group_1 = dict(view_1 = adata))

In [ ]:
MOFA_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("MOFA-FLEX BASE run", i, "of", N_RUNS, "\n")
  
  start_time <- Sys.time()
  
  model <- mfl$MOFAFLEX(
    data_list,
    mfl$DataOptions(
      scale_per_group = FALSE,
      plot_data_overview = FALSE,
      remove_constant_features = TRUE
    ),
    mfl$ModelOptions(
      n_factors = as.integer(K),
      likelihoods = "Normal",
      weight_prior = "Laplace"
    ),
    mfl$TrainingOptions(
      seed = 1L,
      max_epochs = 2000L,
      batch_size = 1000L
    )
  )
  
  end_time <- Sys.time()
  MOFA_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", MOFA_times[i], "minutes\n\n")
}

In [ ]:
MOFA_FLEX_BASE_time_minutes <- MOFA_times
names(MOFA_FLEX_BASE_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(MOFA_FLEX_BASE_time_minutes, file.path(output_dir, "MOFA_FLEX_BASE_time_minutes.rds"))
cat("MOFA-FLEX BASE times:", MOFA_FLEX_BASE_time_minutes, "minutes\n")